<div align="center">

  <h1><img src="https://raw.githubusercontent.com/auxeno/ion/main/assets/logo.png" alt="Ion" width="72"><br>DQN on Atari</h1>

  [![Open in nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/Auxeno/ion/blob/main/examples/dqn_gymnasium.ipynb)
  [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auxeno/ion/blob/main/examples/dqn_gymnasium.ipynb)

</div>

---

Training a DQN agent on [Atari Breakout](https://ale.farama.org/environments/breakout/) using [Ion](https://github.com/auxeno/ion).

In [1]:
# !pip install ion-nn optax gymnasium ale-py opencv-python-headless tqdm matplotlib pygame

In [2]:
from collections import deque
from functools import partial
from typing import Callable, NamedTuple

import ale_py
import gymnasium as gym
import jax
import jax.numpy as jnp
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import optax
from IPython.display import HTML
from jaxtyping import Array, Float, Int, PRNGKeyArray
from tqdm.auto import tqdm

import ion
from ion import nn

gym.register_envs(ale_py)
split = jax.jit(jax.random.split, static_argnums=1)

## Dueling Q-Network

The Q-network estimates action values $Q(s, a)$ for each state action pair. In the dueling architecture, the final layer splits into a **value stream** $V(s)$ and an **advantage stream** $A(s, a)$, recombined as:

$$Q(s, a) = V(s) + \left(A(s, a) - \frac{1}{|\mathcal{A}|}\sum_{a'} A(s, a')\right)$$

Mean-centering the advantages ensures identifiability: the value stream uniquely represents the state value.

In [3]:
class QNetwork(nn.Module):
    """Dueling Q-network for estimating Q-values."""

    torso: nn.MLP
    advantage_head: nn.Linear
    value_head: nn.Linear

    def __init__(self, obs_dim: int, action_dim: int, dim: int = 64, *, key: PRNGKeyArray) -> None:
        key_torso, key_a, key_v = jax.random.split(key, 3)
        self.torso = nn.MLP(obs_dim, dim, dim, num_hidden_layers=2, key=key_torso)
        self.advantage_head = nn.Linear(dim, action_dim, key=key_a)
        self.value_head = nn.Linear(dim, 1, key=key_v)

    def __call__(self, observations: Float[Array, "... d"]) -> Float[Array, "... a"]:
        x = jax.nn.relu(self.torso(observations))
        advantages = self.advantage_head(x)
        value = self.value_head(x)
        q = value + (advantages - advantages.mean(axis=-1, keepdims=True))
        return q

## Replay Buffer

DQN stores transitions in a circular replay buffer and samples random minibatches for learning. This breaks temporal correlations in the training data.

The key JAX trick here is `donate_argnums`: when pushing to the buffer, we tell JAX that the old buffer won't be used again, so it can reuse the memory in-place rather than allocating a new copy. For a 100k-entry buffer this avoids doubling memory on every push.

In [4]:
class Transition(NamedTuple):
    observations: Array
    next_observations: Array
    rewards: Float[Array, "..."]
    terminations: Float[Array, "..."]
    truncations: Float[Array, "..."]
    actions: Int[Array, "..."]


class BufferState(NamedTuple):
    """Circular replay buffer for storing transitions."""

    transitions: Transition
    idx: Int[Array, ""]
    size: Int[Array, ""]


def init_buffer(
    obs_shape: tuple[int, ...], buffer_size: int, obs_dtype: jnp.dtype = jnp.float32
) -> BufferState:
    """Allocate an empty replay buffer."""
    return BufferState(
        transitions=Transition(
            observations=jnp.zeros((buffer_size, *obs_shape), dtype=obs_dtype),
            next_observations=jnp.zeros((buffer_size, *obs_shape), dtype=obs_dtype),
            rewards=jnp.zeros(buffer_size),
            terminations=jnp.zeros(buffer_size),
            truncations=jnp.zeros(buffer_size),
            actions=jnp.zeros(buffer_size, dtype=jnp.int32),
        ),
        idx=jnp.array(0, dtype=jnp.int32),
        size=jnp.array(0, dtype=jnp.int32),
    )


@partial(jax.jit, donate_argnums=(0,))
def buffer_push(buffer: BufferState, experience: tuple[Transition, ...]) -> BufferState:
    """Write rollout experiences into replay buffer."""
    batch = jax.tree.map(lambda *x: jnp.concatenate(x), *experience)
    num_steps = batch.observations.shape[0]
    indices = (buffer.idx + jnp.arange(num_steps)) % BUFFER_SIZE
    new_transitions = jax.tree.map(lambda buf, t: buf.at[indices].set(t), buffer.transitions, batch)
    return BufferState(
        transitions=new_transitions,
        idx=(buffer.idx + num_steps) % BUFFER_SIZE,
        size=jnp.minimum(buffer.size + num_steps, BUFFER_SIZE),
    )

## Action Selection, Rollout and Learning

**Action selection** uses epsilon-greedy: with probability $\epsilon$ take a random action, otherwise take the greedy action $\arg\max_a Q(s, a)$.

**Rollout** collects transitions from vectorized gymnasium environments. Since gymnasium is not JAX-native, the rollout loop runs in Python while action selection is JIT-compiled.

**Learning** implements Double DQN: the online network selects the best next action, but the target network evaluates its Q-value. This reduces the overestimation bias of standard DQN. After each gradient step, the target network is updated via Polyak averaging: $\theta_{\text{target}} \leftarrow \tau \theta_{\text{online}} + (1 - \tau) \theta_{\text{target}}$.

In [5]:
@jax.jit
def select_action(
    network: QNetwork,
    observations: Float[Array, "n d"],
    epsilon: Float[Array, ""],
    *,
    key: PRNGKeyArray,
) -> Int[Array, " n"]:
    """Epsilon-greedy action selection."""
    key_random, key_choice = jax.random.split(key)
    num_envs = observations.shape[0]
    greedy_actions = network(observations.astype(jnp.float32)).argmax(axis=-1)
    random_actions = jax.random.randint(key_random, (num_envs,), 0, ACTION_DIM)
    use_random = jax.random.uniform(key_choice, (num_envs,)) < epsilon
    return jnp.where(use_random, random_actions, greedy_actions)


def rollout(
    network: QNetwork,
    envs: gym.vector.AsyncVectorEnv,
    initial_observations: np.ndarray,
    epsilon: float,
    current_returns: np.ndarray,
    recent_returns: deque[float],
    *,
    key: PRNGKeyArray,
) -> tuple[np.ndarray, tuple[Transition, ...]]:
    """Collect transitions from vectorized environments."""
    experience = []

    observations = initial_observations
    for _ in range(ROLLOUT_STEPS):
        key, key_action = split(key)
        actions = np.asarray(
            select_action(network, jnp.asarray(observations), jnp.float32(epsilon), key=key_action)
        )

        next_observations, rewards, terminations, truncations, _ = envs.step(actions)

        experience.append(
            Transition(
                observations=observations,  # type: ignore[arg-type]
                next_observations=next_observations,  # type: ignore[arg-type]
                rewards=rewards,
                terminations=terminations.astype(np.float32),
                truncations=truncations.astype(np.float32),
                actions=actions,  # type: ignore[arg-type]
            )
        )

        # Track episode returns and reset done environments
        current_returns[:] += rewards
        dones = terminations | truncations
        if np.any(dones):
            for ret in current_returns[dones]:
                recent_returns.append(float(ret))
            current_returns[dones] = 0.0
            next_observations, _ = envs.reset(options={"reset_mask": dones})

        observations = next_observations

    return observations, tuple(experience)


@jax.jit
def learn(
    network: QNetwork,
    target_network: QNetwork,
    optimizer: ion.Optimizer,
    buffer: BufferState,
    *,
    key: PRNGKeyArray,
) -> tuple[QNetwork, QNetwork, ion.Optimizer]:
    """Double DQN update with soft target network update."""
    indices = jax.random.randint(key, (BATCH_SIZE,), 0, buffer.size)
    batch = jax.tree.map(lambda x: x[indices], buffer.transitions)

    # Cast observations to float32 (buffer may store uint8 for memory efficiency)
    observations = batch.observations.astype(jnp.float32)
    next_observations = batch.next_observations.astype(jnp.float32)

    # Double DQN bootstrap targets
    next_actions = network(next_observations).argmax(axis=-1)
    next_q = target_network(next_observations)[jnp.arange(BATCH_SIZE), next_actions]
    targets = batch.rewards + GAMMA * (1.0 - batch.terminations) * next_q

    def loss_fn(network: QNetwork) -> Float[Array, ""]:
        q_values = network(observations)[jnp.arange(BATCH_SIZE), batch.actions]
        return ((targets - q_values) ** 2).mean()

    grads = jax.grad(loss_fn)(network)
    network, optimizer = optimizer.update(network, grads)

    # Polyak soft update target network
    target_network = jax.tree.map(lambda t, o: TAU * o + (1.0 - TAU) * t, target_network, network)
    return network, target_network, optimizer

In [6]:
def train_dqn(
    network: QNetwork,
    env_fn: Callable[[], gym.Env],
    *,
    seed: int = 42,
) -> QNetwork:
    """Train a DQN agent on a gymnasium environment."""

    steps_per_rollout = ROLLOUT_STEPS * NUM_ENVS
    total_rollouts = TOTAL_STEPS // steps_per_rollout

    # Create vectorized environments (we manually reset)
    envs = gym.vector.AsyncVectorEnv(
        [env_fn for _ in range(NUM_ENVS)],
        autoreset_mode="Disabled",
    )

    # Initialize target network and optimizer
    rng = jax.random.key(seed)
    target_network = network
    optimizer = ion.Optimizer(
        optax.chain(optax.clip_by_global_norm(1.0), optax.adam(learning_rate=LR)),
        network,
    )

    # Initialize replay buffer
    obs_shape = envs.single_observation_space.shape  # type: ignore[union-attr]
    obs_dtype = envs.single_observation_space.dtype  # type: ignore[union-attr]
    buffer = init_buffer(obs_shape, BUFFER_SIZE, obs_dtype)  # type: ignore[arg-type]
    observations, _ = envs.reset(seed=seed)

    # Episode tracking
    current_returns = np.zeros(NUM_ENVS)
    recent_returns: deque[float] = deque(maxlen=100)
    steps = 0

    bar = tqdm(total=TOTAL_STEPS, desc="DQN")
    for rollout_idx in range(total_rollouts):
        rng, key_rollout, key_learn = split(rng, 3)

        epsilon = max(
            EPS_FINAL,
            EPS_START - (EPS_START - EPS_FINAL) * steps / (TOTAL_STEPS * EPS_FRACTION),
        )

        observations, experience = rollout(
            network, envs, observations, epsilon, current_returns, recent_returns, key=key_rollout
        )
        buffer = buffer_push(buffer, experience)

        steps += steps_per_rollout
        bar.update(steps_per_rollout)

        if steps >= LEARNING_STARTS:
            network, target_network, optimizer = learn(
                network, target_network, optimizer, buffer, key=key_learn
            )

        if recent_returns and steps % 1000 < steps_per_rollout:
            mean_reward = np.mean(recent_returns)
            bar.set_postfix(reward=f"{mean_reward:.1f}", eps=f"{epsilon:.2f}")

    bar.close()
    envs.close()
    return network

## CartPole

Quick sanity check on CartPole-v1 (max reward 500). Trains in ~30 seconds on CPU, ~2 mins on GPU.

In [7]:
TOTAL_STEPS = 500_000
ROLLOUT_STEPS = 4
NUM_ENVS = 8
LR = 1e-3
GAMMA = 0.99
TAU = 0.05
BUFFER_SIZE = 100_000
BATCH_SIZE = 128
LEARNING_STARTS = 1_000
EPS_START = 1.0
EPS_FINAL = 0.05
EPS_FRACTION = 0.5
SEED = 42

sample_env = gym.make("CartPole-v1")
OBS_DIM = sample_env.observation_space.shape[0]  # type: ignore[index]
ACTION_DIM = int(sample_env.action_space.n)  # type: ignore[attr-defined]
sample_env.close()

rng = jax.random.key(SEED)
rng, key_network = jax.random.split(rng)
network = QNetwork(OBS_DIM, ACTION_DIM, key=key_network)

cartpole_network = train_dqn(network, lambda: gym.make("CartPole-v1"), seed=SEED)

DQN: 100%|██████████| 500000/500000 [02:19<00:00, 3793.65it/s, eps=0.05, reward=400.7]


## Evaluation

We can evaluate the trained agent by running episodes greedily (no exploration) and recording frames for visualization.

In [8]:
def evaluate(
    network: QNetwork,
    env: gym.Env,
    num_episodes: int = 5,
    *,
    seed: int = 0,
) -> tuple[list[float], list[list[np.ndarray]]]:
    """Evaluate agent greedily, returning episode rewards and frames."""
    episode_rewards = []
    episode_frames = []

    for ep in range(num_episodes):
        obs, _ = env.reset(seed=seed + ep)
        frames = [env.render()]
        total_reward = 0.0

        while True:
            q_values = network(jnp.asarray(obs[None]))[0]
            action = int(q_values.argmax())
            obs, reward, terminated, truncated, _ = env.step(action)
            frames.append(env.render())
            total_reward += float(reward)
            if terminated or truncated:
                break

        episode_rewards.append(total_reward)
        episode_frames.append(frames)

    return episode_rewards, episode_frames


def display_episode(frames: list[np.ndarray], fps: int = 30) -> HTML:
    """Display a list of frames as an embedded video in the notebook."""
    fig, ax = plt.subplots()
    ax.axis("off")
    img = ax.imshow(frames[0])
    plt.close()

    def update(i):
        img.set_data(frames[i])
        return (img,)

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 / fps, blit=True)
    return HTML(anim.to_html5_video())

In [9]:
eval_env = gym.make("CartPole-v1", render_mode="rgb_array")
rewards, frames = evaluate(cartpole_network, eval_env)
eval_env.close()
print(f"Mean reward: {np.mean(rewards):.1f} over {len(rewards)} episodes")
display_episode(frames[0], fps=60)

Mean reward: 470.2 over 5 episodes


## Atari Breakout

Now we scale to pixel observations. The standard Atari preprocessing pipeline:
- **Frame skip** (4): repeat each action for 4 frames, accumulating reward
- **Max-pooling**: element-wise max of last 2 raw frames (handles sprite flickering)
- **Grayscale + resize**: convert to 84x84 grayscale
- **Frame stacking** (4): stack 4 consecutive frames as input channels

This gives observations of shape `(4, 84, 84)` as uint8. We store them as-is in the replay buffer (8x smaller than float32) and normalize to [0, 1] inside the network's forward pass. The only change needed to our DQN code is a CNN-based Q-network -- everything else (buffer, rollout, learning) works unchanged.

In [10]:
def make_atari(env_id: str = "ALE/Breakout-v5") -> gym.Env:
    """Create an Atari environment with standard DQN preprocessing."""
    env = gym.make(env_id, frameskip=1, repeat_action_probability=0.0)
    env = gym.wrappers.AtariPreprocessing(
        env,
        noop_max=10,
        frame_skip=4,
        terminal_on_life_loss=True,
        grayscale_obs=True,
        scale_obs=False,
    )
    env = gym.wrappers.FrameStackObservation(env, stack_size=4)
    return env

### Nature CNN Q-Network

The [Nature DQN](https://www.nature.com/articles/nature14236) CNN architecture: three convolutional layers followed by a fully connected layer, then the dueling value/advantage heads.

Ion's `Conv` uses channels-last format `(H, W, C)`, so we transpose the `(4, 84, 84)` input to `(84, 84, 4)`.

In [ ]:
class AtariQNetwork(nn.Module):
    """Nature CNN dueling Q-network for pixel observations."""

    conv1: nn.Conv
    conv2: nn.Conv
    conv3: nn.Conv
    fc: nn.Linear
    advantage_head: nn.Linear
    value_head: nn.Linear

    def __init__(self, action_dim: int, dim: int = 512, *, key: PRNGKeyArray) -> None:
        k1, k2, k3, k4, k5, k6 = jax.random.split(key, 6)
        self.conv1 = nn.Conv(4, 32, kernel_shape=(8, 8), stride=(4, 4), key=k1)
        self.conv2 = nn.Conv(32, 64, kernel_shape=(4, 4), stride=(2, 2), key=k2)
        self.conv3 = nn.Conv(64, 64, kernel_shape=(3, 3), stride=(1, 1), key=k3)
        self.fc = nn.Linear(3136, dim, key=k4)  # 7 * 7 * 64
        self.advantage_head = nn.Linear(dim, action_dim, key=k5)
        self.value_head = nn.Linear(dim, 1, key=k6)

    def __call__(self, observations: Float[Array, "... c h w"]) -> Float[Array, "... a"]:
        # Normalize uint8 -> [0, 1] and transpose to channels-last for Ion Conv
        x = observations / 255.0
        x = jnp.moveaxis(x, -3, -1)
        x = jax.nn.relu(self.conv1(x))
        x = jax.nn.relu(self.conv2(x))
        x = jax.nn.relu(self.conv3(x))
        x = x.reshape(*x.shape[:-3], -1)  # flatten spatial dims
        x = jax.nn.relu(self.fc(x))
        advantages = self.advantage_head(x)
        value = self.value_head(x)
        return value + (advantages - advantages.mean(axis=-1, keepdims=True))

### Train on Breakout

We use the same `train_dqn` function with adjusted hyperparameters for Atari. The only things that change are the environment factory, the network, and the constants.

In [12]:
TOTAL_STEPS = 1_000_000
ROLLOUT_STEPS = 1
NUM_ENVS = 16
LR = 1e-3
GAMMA = 0.99
TAU = 0.005
BUFFER_SIZE = 500_000
BATCH_SIZE = 128
LEARNING_STARTS = 20_000
EPS_START = 0.8
EPS_FINAL = 0.1
EPS_FRACTION = 0.7
SEED = 42

sample_env = make_atari()
ACTION_DIM = int(sample_env.action_space.n)  # type: ignore[attr-defined]
sample_env.close()

rng = jax.random.key(SEED)
rng, key_network = jax.random.split(rng)
atari_network = AtariQNetwork(ACTION_DIM, key=key_network)

breakout_network = train_dqn(atari_network, make_atari, seed=SEED)

DQN: 100%|██████████| 1000000/1000000 [10:21<00:00, 1924.73it/s, eps=0.10, reward=7.6]


### Visualize Breakout Agent

In [ ]:
def make_atari_eval(env_id: str = "ALE/Breakout-v5") -> gym.Env:
    """Create an Atari environment for evaluation with rgb rendering."""
    env = gym.make(env_id, frameskip=1, repeat_action_probability=0.0, render_mode="rgb_array")
    env = gym.wrappers.AtariPreprocessing(env, frame_skip=4, grayscale_obs=True, scale_obs=False)
    env = gym.wrappers.FrameStackObservation(env, stack_size=4)
    return env


eval_env = make_atari_eval()
rewards, frames = evaluate(breakout_network, eval_env, num_episodes=2)
eval_env.close()
print(f"Mean reward: {np.mean(rewards):.1f} over {len(rewards)} episodes")
display_episode(frames[0], fps=30)

Mean reward: 92.5 over 2 episodes
